# 🧠 Classic RAG vs PageIndex: A Comparative Study

> **"Semantic similarity is not enough for structured documents"**

This notebook demonstrates two RAG strategies on a structured document (Hospital SRS):

1. **Classic RAG** — Vector-based retrieval using cosine similarity
2. **PageIndex RAG** — Hierarchical tree navigation using LLM reasoning

**LLM used:** `llama-3.1-8b-instant` via **Groq API** (ultra-fast inference, free tier available)

**Embeddings:** `all-MiniLM-L6-v2` via Sentence-Transformers (local, free)

---
**Author:** [Rozêra Xelîl]  
**Published on:** Medium  
**GitHub:** []

## 📦 Step 1: Install Dependencies

In [ ]:
# Install all required packages
# Run this cell once — restart kernel after installation if needed
!pip install -q faiss-cpu sentence-transformers numpy requests

## 📚 Step 2: Imports & Configuration

In [1]:
import json
import time
import re
import numpy as np
import faiss
import requests
from typing import Optional
from sentence_transformers import SentenceTransformer

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## ⚙️ Step 3: Groq LLM Client (Production-Grade)

We encapsulate the Groq API into a clean, reusable class with:
- ✅ Automatic retry on transient failures
- ✅ Configurable temperature & max_tokens
- ✅ Structured error handling
- ✅ Usage tracking

In [ ]:
class GroqLLMClient:
    """
    Production-grade Groq API client.

    Wraps the Groq OpenAI-compatible endpoint with:
    - Retry logic with exponential backoff
    - Structured error reporting
    - Usage statistics tracking
    """

    BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

    def __init__(
        self,
        api_key: str,
        model: str = "llama-3.1-8b-instant",
        max_retries: int = 3,
        retry_delay: float = 1.0,
    ):
        """
        Args:
            api_key:      Your Groq API key.
            model:        Model ID (default: llama-3.1-8b-instant).
            max_retries:  Max retry attempts on failure.
            retry_delay:  Base delay in seconds between retries (exponential backoff).
        """
        self.api_key = api_key
        self.model = model
        self.max_retries = max_retries
        self.retry_delay = retry_delay

        self._headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

        # Track cumulative token usage across all calls
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_calls = 0

    # ──────────────────────────────────────────────────────────────
    # Public interface
    # ──────────────────────────────────────────────────────────────

    def complete(
        self,
        prompt: str,
        system: str = "You are a helpful, precise assistant.",
        max_tokens: int = 512,
        temperature: float = 0.0,
    ) -> str:
        """
        Send a single-turn completion request to Groq.

        Args:
            prompt:      User-facing prompt text.
            system:      System instruction for the model.
            max_tokens:  Maximum tokens to generate.
            temperature: Sampling temperature (0 = deterministic).

        Returns:
            Generated text string.

        Raises:
            RuntimeError: If all retries are exhausted.
        """
        payload = {
            "model": self.model,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user",   "content": prompt},
            ],
        }

        for attempt in range(1, self.max_retries + 1):
            try:
                response = requests.post(
                    self.BASE_URL,
                    headers=self._headers,
                    json=payload,
                    timeout=30,
                )

                if response.status_code == 200:
                    data = response.json()
                    self._track_usage(data.get("usage", {}))
                    return data["choices"][0]["message"]["content"].strip()

                elif response.status_code == 429:
                    # Rate limit — back off and retry
                    wait = self.retry_delay * (2 ** (attempt - 1))
                    print(f"  ⚠️  Rate limit hit. Waiting {wait:.1f}s before retry {attempt}/{self.max_retries}...")
                    time.sleep(wait)

                else:
                    raise RuntimeError(
                        f"Groq API error {response.status_code}: {response.text}"
                    )

            except requests.exceptions.Timeout:
                print(f"  ⚠️  Request timeout on attempt {attempt}/{self.max_retries}.")
                if attempt == self.max_retries:
                    raise RuntimeError("All retries exhausted due to repeated timeouts.")
                time.sleep(self.retry_delay * attempt)

            except requests.exceptions.ConnectionError as exc:
                raise RuntimeError(f"Network error: {exc}") from exc

        raise RuntimeError(f"All {self.max_retries} retries exhausted.")

    def usage_report(self) -> dict:
        """Return a summary of cumulative token usage."""
        return {
            "total_calls":             self.total_calls,
            "total_prompt_tokens":     self.total_prompt_tokens,
            "total_completion_tokens": self.total_completion_tokens,
            "total_tokens":            self.total_prompt_tokens + self.total_completion_tokens,
        }

    # ──────────────────────────────────────────────────────────────
    # Private helpers
    # ──────────────────────────────────────────────────────────────

    def _track_usage(self, usage: dict) -> None:
        self.total_calls += 1
        self.total_prompt_tokens     += usage.get("prompt_tokens",     0)
        self.total_completion_tokens += usage.get("completion_tokens", 0)


# ── Instantiate the client ─────────────────────────────────────────
GROQ_API_KEY = "*"  # ← replace if needed

llm = GroqLLMClient(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
)

# Smoke test
test_response = llm.complete(
    prompt="Reply with exactly: GROQ_OK",
    max_tokens=10,
)
print(f"✅ Groq client initialised — model: {llm.model}")
print(f"   Smoke test response: {test_response}")

✅ Groq client initialised — model: llama-3.1-8b-instant
   Smoke test response: GROQ_OK


## 🏥 Step 4: Sample Document — Hospital SRS

We simulate a structured Hospital Software Requirements Specification (SRS) document.  
This represents a **real-world hierarchical document** — the kind that breaks Classic RAG.

In [3]:
# ─────────────────────────────────────────────
# Simulated Hospital SRS Document
# ─────────────────────────────────────────────

DOCUMENT = """
Chapter 1: Introduction
Section 1.1: Purpose
This document describes the software requirements for the Hospital Management System (HMS).
The system is designed to streamline hospital operations including patient registration,
doctor scheduling, billing, and medical record management.

Section 1.2: Scope
The HMS will serve all departments including emergency, outpatient, inpatient, and pharmacy.
It targets hospital administrators, doctors, nurses, and patients.

Chapter 2: System Overview
Section 2.1: System Architecture
The system uses a three-tier architecture: Presentation Layer (React frontend),
Business Logic Layer (Node.js backend), and Data Layer (PostgreSQL database).
All components communicate via RESTful APIs.

Section 2.2: User Roles
The system defines four user roles: Admin, Doctor, Nurse, and Patient.
Each role has specific permissions and access levels defined in the RBAC module.

Chapter 3: Security Requirements
Section 3.1: Data Encryption
All patient data must be encrypted using AES-256 encryption at rest.
Data in transit must use TLS 1.3 protocol. Encryption keys are managed
by a dedicated Key Management Service (KMS).

Section 3.2: Authentication
The system uses JWT-based authentication with OAuth 2.0 for third-party integrations.
Multi-Factor Authentication (MFA) is mandatory for all admin and doctor accounts.
Session tokens expire after 30 minutes of inactivity.

Section 3.3: Audit Logging
All user actions must be logged in an immutable audit trail.
Logs include: user ID, timestamp, action type, and affected resource.
Audit logs are retained for a minimum of 7 years per HIPAA regulations.

Chapter 4: Functional Requirements
Section 4.1: Patient Registration
Patients can register via web portal or in-person kiosk.
Required fields: full name, date of birth, national ID, insurance number, contact info.
Duplicate detection is performed using national ID and date of birth matching.

Section 4.2: Appointment Scheduling
Patients can book, reschedule, or cancel appointments online.
The system sends automated reminders via SMS and email 24 hours before appointments.
Emergency appointments bypass the regular queue system.

Section 4.3: Billing and Payments
The billing module generates invoices automatically after each visit.
Supported payment methods: credit card, insurance claim, and government subsidy.
All transactions are logged and reconciled daily.

Chapter 5: Non-Functional Requirements
Section 5.1: Performance
The system must support 10,000 concurrent users with a response time under 2 seconds.
Database queries must be optimized to complete in under 500ms.

Section 5.2: Availability
The system must achieve 99.9% uptime (less than 9 hours downtime per year).
Planned maintenance windows are limited to Sunday 2:00 AM - 4:00 AM.
"""

print(f"📄 Document loaded: {len(DOCUMENT.split())} words, {len(DOCUMENT.splitlines())} lines")

📄 Document loaded: 394 words, 61 lines


## 🔵 PIPELINE 1: Classic RAG (Vector-Based)

```
Document → Chunking → Embeddings → FAISS Index → Similarity Search → LLM → Answer
```

### 1A: Chunking the Document

In [4]:
def chunk_document(text: str, chunk_size: int = 100, overlap: int = 20) -> list[str]:
    """
    Split document into overlapping word-based chunks.

    Args:
        text:       Raw document string.
        chunk_size: Words per chunk.
        overlap:    Overlap between consecutive chunks.

    Returns:
        List of chunk strings.
    """
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


chunks = chunk_document(DOCUMENT)
print(f"✂️  Document split into {len(chunks)} chunks")
print("\n📌 Sample chunk:")
print("-" * 60)
print(chunks[2])

✂️  Document split into 5 chunks

📌 Sample chunk:
------------------------------------------------------------
3.2: Authentication The system uses JWT-based authentication with OAuth 2.0 for third-party integrations. Multi-Factor Authentication (MFA) is mandatory for all admin and doctor accounts. Session tokens expire after 30 minutes of inactivity. Section 3.3: Audit Logging All user actions must be logged in an immutable audit trail. Logs include: user ID, timestamp, action type, and affected resource. Audit logs are retained for a minimum of 7 years per HIPAA regulations. Chapter 4: Functional Requirements Section 4.1: Patient Registration Patients can register via web portal or in-person kiosk. Required fields: full name, date of birth, national ID, insurance number, contact info.


### 1B: Embedding Chunks with Sentence-Transformers

In [5]:
# Load a free, lightweight embedding model from Hugging Face
# all-MiniLM-L6-v2: fast (~80MB), effective, runs on CPU
print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded!")

# Encode all chunks into vectors
print("\n⏳ Encoding chunks...")
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)
print(f"\n✅ Embedding matrix shape: {chunk_embeddings.shape}")
print(f"   → {chunk_embeddings.shape[0]} chunks × {chunk_embeddings.shape[1]} dimensions")

⏳ Loading embedding model (all-MiniLM-L6-v2)...
✅ Embedding model loaded!

⏳ Encoding chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Embedding matrix shape: (5, 384)
   → 5 chunks × 384 dimensions


### 1C: Build FAISS Vector Index

In [6]:
def build_faiss_index(embeddings: np.ndarray) -> tuple:
    """
    Build a FAISS flat L2 index on unit-normalised vectors
    (equivalent to cosine similarity search).

    Returns:
        (index, normalised_embeddings)
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    normed = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    index.add(normed.astype(np.float32))
    return index, normed


faiss_index, normalised_embeddings = build_faiss_index(chunk_embeddings)
print(f"🗂️  FAISS index built with {faiss_index.ntotal} vectors")

🗂️  FAISS index built with 5 vectors


### 1D: Retrieval Function (Classic RAG)

In [7]:
def classic_rag_retrieve(query: str, top_k: int = 3) -> tuple[list[str], np.ndarray]:
    """
    Retrieve top-k most similar chunks using cosine similarity via FAISS.

    Args:
        query: User question string.
        top_k: Number of chunks to retrieve.

    Returns:
        (retrieved_chunks, distances)
    """
    query_vec = embedder.encode([query])
    query_vec = query_vec / np.linalg.norm(query_vec)
    distances, indices = faiss_index.search(query_vec.astype(np.float32), top_k)
    retrieved = [chunks[i] for i in indices[0]]
    return retrieved, distances[0]


# Quick test
test_query = "How is patient data encrypted?"
retrieved, scores = classic_rag_retrieve(test_query)
print(f"🔍 Query: '{test_query}'")
print(f"\n📦 Retrieved {len(retrieved)} chunks (lower L2 distance = more similar):")
for i, (chunk, score) in enumerate(zip(retrieved, scores)):
    print(f"\n[Chunk {i+1}] Score: {score:.4f}")
    print("-" * 50)
    print(chunk[:200] + "...")

🔍 Query: 'How is patient data encrypted?'

📦 Retrieved 3 chunks (lower L2 distance = more similar):

[Chunk 1] Score: 1.1262
--------------------------------------------------
(Node.js backend), and Data Layer (PostgreSQL database). All components communicate via RESTful APIs. Section 2.2: User Roles The system defines four user roles: Admin, Doctor, Nurse, and Patient. Eac...

[Chunk 2] Score: 1.3008
--------------------------------------------------
3.2: Authentication The system uses JWT-based authentication with OAuth 2.0 for third-party integrations. Multi-Factor Authentication (MFA) is mandatory for all admin and doctor accounts. Session toke...

[Chunk 3] Score: 1.3012
--------------------------------------------------
Chapter 1: Introduction Section 1.1: Purpose This document describes the software requirements for the Hospital Management System (HMS). The system is designed to streamline hospital operations includ...


### 1E: Classic RAG — Full Answer Generation via Groq

In [8]:
CLASSIC_RAG_SYSTEM = (
    "You are a precise technical assistant. "
    "Answer questions strictly from the provided context. "
    "Be concise and factual. If the context does not contain the answer, say so."
)


def classic_rag_answer(query: str, top_k: int = 3) -> dict:
    """
    Full Classic RAG pipeline:
      1. Retrieve relevant chunks via FAISS (cosine similarity).
      2. Build prompt with retrieved context.
      3. Generate answer via Groq (llama-3.1-8b-instant).

    Returns:
        dict with keys: query, retrieved_chunks, scores, answer
    """
    # Step 1 — Retrieve
    retrieved_chunks, scores = classic_rag_retrieve(query, top_k)
    context = "\n\n".join(
        f"[Chunk {i+1}]\n{c}" for i, c in enumerate(retrieved_chunks)
    )

    # Step 2 — Build prompt
    prompt = f"""Context (retrieved document excerpts):
─────────────────────────────────────────
{context}
─────────────────────────────────────────

Question: {query}

Answer based only on the context above:"""

    # Step 3 — Generate via Groq
    answer = llm.complete(
        prompt=prompt,
        system=CLASSIC_RAG_SYSTEM,
        max_tokens=300,
        temperature=0.0,
    )

    return {
        "query":            query,
        "retrieved_chunks": retrieved_chunks,
        "scores":           scores,
        "answer":           answer,
    }


# Test it!
result_classic = classic_rag_answer("How is patient data encrypted?")
print("═" * 60)
print("🔵 CLASSIC RAG ANSWER")
print("═" * 60)
print(f"Query   : {result_classic['query']}")
print(f"Chunks  : {len(result_classic['retrieved_chunks'])} retrieved")
print(f"Answer  :\n{result_classic['answer']}")

════════════════════════════════════════════════════════════
🔵 CLASSIC RAG ANSWER
════════════════════════════════════════════════════════════
Query   : How is patient data encrypted?
Chunks  : 3 retrieved
Answer  :
Patient data is encrypted using AES-256 encryption at rest.


---

## 🟢 PIPELINE 2: PageIndex RAG (Structural / Reasoning-Based)

```
Document → Hierarchical Tree Index → LLM Navigation → Target Node → LLM Answer
```

The key insight: instead of searching by **math**, we navigate by **reasoning**.

### 2A: Build the Hierarchical Tree Index

In [9]:
# This is the "brain" of PageIndex.
# In production this would be auto-extracted by an LLM from any document.
# Here we build it manually to focus on the concept.

PAGE_INDEX_TREE = {
    "title": "Hospital Management System — SRS",
    "children": [
        {
            "id": "ch1",
            "title": "Chapter 1: Introduction",
            "children": [
                {
                    "id": "sec1.1",
                    "title": "Section 1.1: Purpose",
                    "summary": "Describes the purpose of HMS: streamlining hospital operations including patient registration, scheduling, billing, and medical records.",
                    "content": "This document describes the software requirements for the Hospital Management System (HMS). The system is designed to streamline hospital operations including patient registration, doctor scheduling, billing, and medical record management."
                },
                {
                    "id": "sec1.2",
                    "title": "Section 1.2: Scope",
                    "summary": "Defines scope: covers emergency, outpatient, inpatient, and pharmacy. Targets admins, doctors, nurses, and patients.",
                    "content": "The HMS will serve all departments including emergency, outpatient, inpatient, and pharmacy. It targets hospital administrators, doctors, nurses, and patients."
                }
            ]
        },
        {
            "id": "ch2",
            "title": "Chapter 2: System Overview",
            "children": [
                {
                    "id": "sec2.1",
                    "title": "Section 2.1: System Architecture",
                    "summary": "Three-tier architecture: React frontend, Node.js backend, PostgreSQL database. RESTful API communication.",
                    "content": "The system uses a three-tier architecture: Presentation Layer (React frontend), Business Logic Layer (Node.js backend), and Data Layer (PostgreSQL database). All components communicate via RESTful APIs."
                },
                {
                    "id": "sec2.2",
                    "title": "Section 2.2: User Roles",
                    "summary": "Four roles defined: Admin, Doctor, Nurse, Patient. Permissions managed via RBAC module.",
                    "content": "The system defines four user roles: Admin, Doctor, Nurse, and Patient. Each role has specific permissions and access levels defined in the RBAC module."
                }
            ]
        },
        {
            "id": "ch3",
            "title": "Chapter 3: Security Requirements",
            "children": [
                {
                    "id": "sec3.1",
                    "title": "Section 3.1: Data Encryption",
                    "summary": "Patient data encrypted with AES-256 at rest. TLS 1.3 in transit. Keys managed by KMS.",
                    "content": "All patient data must be encrypted using AES-256 encryption at rest. Data in transit must use TLS 1.3 protocol. Encryption keys are managed by a dedicated Key Management Service (KMS)."
                },
                {
                    "id": "sec3.2",
                    "title": "Section 3.2: Authentication",
                    "summary": "JWT + OAuth 2.0 authentication. MFA mandatory for admins and doctors. Sessions expire after 30 minutes.",
                    "content": "The system uses JWT-based authentication with OAuth 2.0 for third-party integrations. Multi-Factor Authentication (MFA) is mandatory for all admin and doctor accounts. Session tokens expire after 30 minutes of inactivity."
                },
                {
                    "id": "sec3.3",
                    "title": "Section 3.3: Audit Logging",
                    "summary": "Immutable audit trail for all actions. Logs retained 7 years per HIPAA. Fields: user ID, timestamp, action, resource.",
                    "content": "All user actions must be logged in an immutable audit trail. Logs include: user ID, timestamp, action type, and affected resource. Audit logs are retained for a minimum of 7 years per HIPAA regulations."
                }
            ]
        },
        {
            "id": "ch4",
            "title": "Chapter 4: Functional Requirements",
            "children": [
                {
                    "id": "sec4.1",
                    "title": "Section 4.1: Patient Registration",
                    "summary": "Registration via web or kiosk. Required fields: name, DOB, national ID, insurance, contact. Duplicate detection by national ID + DOB.",
                    "content": "Patients can register via web portal or in-person kiosk. Required fields: full name, date of birth, national ID, insurance number, contact info. Duplicate detection is performed using national ID and date of birth matching."
                },
                {
                    "id": "sec4.2",
                    "title": "Section 4.2: Appointment Scheduling",
                    "summary": "Online booking, rescheduling, cancellation. SMS/email reminders 24h before. Emergency appointments bypass queue.",
                    "content": "Patients can book, reschedule, or cancel appointments online. The system sends automated reminders via SMS and email 24 hours before appointments. Emergency appointments bypass the regular queue system."
                },
                {
                    "id": "sec4.3",
                    "title": "Section 4.3: Billing and Payments",
                    "summary": "Auto-invoice after visits. Supports credit card, insurance claim, government subsidy. Daily reconciliation.",
                    "content": "The billing module generates invoices automatically after each visit. Supported payment methods: credit card, insurance claim, and government subsidy. All transactions are logged and reconciled daily."
                }
            ]
        },
        {
            "id": "ch5",
            "title": "Chapter 5: Non-Functional Requirements",
            "children": [
                {
                    "id": "sec5.1",
                    "title": "Section 5.1: Performance",
                    "summary": "Supports 10,000 concurrent users. Response time under 2 seconds. DB queries under 500ms.",
                    "content": "The system must support 10,000 concurrent users with a response time under 2 seconds. Database queries must be optimized to complete in under 500ms."
                },
                {
                    "id": "sec5.2",
                    "title": "Section 5.2: Availability",
                    "summary": "99.9% uptime required. Max 9 hours downtime per year. Maintenance only Sundays 2-4 AM.",
                    "content": "The system must achieve 99.9% uptime (less than 9 hours downtime per year). Planned maintenance windows are limited to Sunday 2:00 AM - 4:00 AM."
                }
            ]
        }
    ]
}

print("🌳 PageIndex Tree built!")
print(f"   Chapters : {len(PAGE_INDEX_TREE['children'])}")
total_sections = sum(len(ch["children"]) for ch in PAGE_INDEX_TREE["children"])
print(f"   Sections : {total_sections}")

🌳 PageIndex Tree built!
   Chapters : 5
   Sections : 12


### 2B: Flatten Tree into a Lookup Map

In [10]:
def flatten_tree(tree: dict) -> dict:
    """
    Flatten the hierarchical tree into {section_id: node} for O(1) lookup.
    """
    node_map = {}
    for chapter in tree["children"]:
        for section in chapter.get("children", []):
            node_map[section["id"]] = section
    return node_map


NODE_MAP = flatten_tree(PAGE_INDEX_TREE)

print("🗺️  Node map created:")
for node_id, node in NODE_MAP.items():
    print(f"  [{node_id}] {node['title']}")

🗺️  Node map created:
  [sec1.1] Section 1.1: Purpose
  [sec1.2] Section 1.2: Scope
  [sec2.1] Section 2.1: System Architecture
  [sec2.2] Section 2.2: User Roles
  [sec3.1] Section 3.1: Data Encryption
  [sec3.2] Section 3.2: Authentication
  [sec3.3] Section 3.3: Audit Logging
  [sec4.1] Section 4.1: Patient Registration
  [sec4.2] Section 4.2: Appointment Scheduling
  [sec4.3] Section 4.3: Billing and Payments
  [sec5.1] Section 5.1: Performance
  [sec5.2] Section 5.2: Availability


### 2C: Build Navigation Prompt (The "Brain" of PageIndex)

In [11]:
def build_navigation_prompt(query: str, tree: dict) -> str:
    """
    Build the navigation prompt that asks the LLM:
    'Which section ID in this tree best answers the question?'

    This is the reasoning step — the core of PageIndex.
    """
    index_lines = []
    for chapter in tree["children"]:
        index_lines.append(f"\n{chapter['title']}")
        for section in chapter.get("children", []):
            index_lines.append(
                f"  [{section['id']}] {section['title']}: {section['summary']}"
            )

    index_text = "\n".join(index_lines)
    valid_ids  = ", ".join(NODE_MAP.keys())

    prompt = f"""Document Structure:
{index_text}

Valid section IDs: {valid_ids}

Question: {query}

Which single section ID contains the most relevant answer?
Reply with ONLY the section ID. Nothing else."""

    return prompt


NAVIGATION_SYSTEM = (
    "You are a document navigator. "
    "Your only job is to identify which section ID in the document "
    "contains the answer to a question. "
    "Reply with ONLY the section ID (e.g., sec3.1). No explanation."
)

# Preview the prompt
sample_prompt = build_navigation_prompt("How is patient data encrypted?", PAGE_INDEX_TREE)
print("📋 Navigation Prompt Preview:")
print("=" * 60)
print(sample_prompt)

📋 Navigation Prompt Preview:
Document Structure:

Chapter 1: Introduction
  [sec1.1] Section 1.1: Purpose: Describes the purpose of HMS: streamlining hospital operations including patient registration, scheduling, billing, and medical records.
  [sec1.2] Section 1.2: Scope: Defines scope: covers emergency, outpatient, inpatient, and pharmacy. Targets admins, doctors, nurses, and patients.

Chapter 2: System Overview
  [sec2.1] Section 2.1: System Architecture: Three-tier architecture: React frontend, Node.js backend, PostgreSQL database. RESTful API communication.
  [sec2.2] Section 2.2: User Roles: Four roles defined: Admin, Doctor, Nurse, Patient. Permissions managed via RBAC module.

Chapter 3: Security Requirements
  [sec3.1] Section 3.1: Data Encryption: Patient data encrypted with AES-256 at rest. TLS 1.3 in transit. Keys managed by KMS.
  [sec3.2] Section 3.2: Authentication: JWT + OAuth 2.0 authentication. MFA mandatory for admins and doctors. Sessions expire after 30 minutes.


### 2D: Navigate + Retrieve + Answer (Full PageIndex Pipeline)

In [12]:
ANSWER_SYSTEM = (
    "You are a technical documentation assistant. "
    "Answer the question using ONLY the provided context. "
    "Be precise and concise. Cite specific details from the context."
)


def extract_node_id(raw: str) -> Optional[str]:
    """
    Robustly extract a valid section ID from the LLM's raw navigation output.
    Handles cases where the model adds extra text.
    """
    # Try exact match first
    raw_stripped = raw.strip().lower()
    for key in NODE_MAP:
        if key in raw_stripped:
            return key
    # Regex fallback: look for pattern secN.N
    match = re.search(r"sec\d+\.\d+", raw_stripped)
    if match and match.group() in NODE_MAP:
        return match.group()
    return None


def keyword_fallback(query: str) -> str:
    """Last-resort: find best matching node by keyword overlap with summaries."""
    query_words = set(w for w in query.lower().split() if len(w) > 4)
    best_id, best_score = list(NODE_MAP.keys())[0], 0
    for node_id, node in NODE_MAP.items():
        summary_words = set(node["summary"].lower().split())
        score = len(query_words & summary_words)
        if score > best_score:
            best_score, best_id = score, node_id
    return best_id


def pageindex_rag_answer(query: str) -> dict:
    """
    Full PageIndex pipeline:

    Step 1 — Navigation : Ask Groq LLM which tree node answers the query.
    Step 2 — Retrieval  : Fetch exact content from that node (O(1) lookup).
    Step 3 — Generation : Generate a focused answer from the targeted context.

    Returns:
        dict with keys: query, navigated_to, node_title, node_summary,
                        context_used, answer, nav_fallback_used
    """
    nav_fallback = False

    # ── STEP 1: NAVIGATION ──────────────────────────────────────
    nav_prompt = build_navigation_prompt(query, PAGE_INDEX_TREE)
    raw_nav    = llm.complete(
        prompt=nav_prompt,
        system=NAVIGATION_SYSTEM,
        max_tokens=20,
        temperature=0.0,
    )

    node_id = extract_node_id(raw_nav)

    if node_id is None:
        node_id      = keyword_fallback(query)
        nav_fallback = True

    # ── STEP 2: RETRIEVAL ────────────────────────────────────────
    target_node = NODE_MAP[node_id]
    context     = target_node["content"]

    # ── STEP 3: GENERATION ──────────────────────────────────────
    answer_prompt = f"""Context (Section {node_id} — {target_node['title']}):
─────────────────────────────────────────
{context}
─────────────────────────────────────────

Question: {query}

Answer based only on the context above:"""

    answer = llm.complete(
        prompt=answer_prompt,
        system=ANSWER_SYSTEM,
        max_tokens=300,
        temperature=0.0,
    )

    return {
        "query":            query,
        "navigated_to":     node_id,
        "node_title":       target_node["title"],
        "node_summary":     target_node["summary"],
        "context_used":     context,
        "answer":           answer,
        "nav_fallback_used": nav_fallback,
    }


# Test it!
result_pageindex = pageindex_rag_answer("How is patient data encrypted?")
print("═" * 60)
print("🟢 PAGEINDEX RAG ANSWER")
print("═" * 60)
print(f"Query         : {result_pageindex['query']}")
print(f"Navigated to  : {result_pageindex['navigated_to']} — {result_pageindex['node_title']}")
print(f"Node Summary  : {result_pageindex['node_summary']}")
print(f"Answer        :\n{result_pageindex['answer']}")
if result_pageindex["nav_fallback_used"]:
    print("⚠️  Note: Keyword fallback used for navigation.")

════════════════════════════════════════════════════════════
🟢 PAGEINDEX RAG ANSWER
════════════════════════════════════════════════════════════
Query         : How is patient data encrypted?
Navigated to  : sec3.1 — Section 3.1: Data Encryption
Node Summary  : Patient data encrypted with AES-256 at rest. TLS 1.3 in transit. Keys managed by KMS.
Answer        :
Patient data is encrypted using AES-256 encryption at rest.


---

## ⚖️ Step 5: Full Comparison Experiment

We run **both pipelines** on multiple questions and compare results side-by-side.

In [14]:
# Test questions designed to challenge structural understanding
TEST_QUESTIONS = [
    "How is patient data encrypted?",
    "What authentication method does the system use?",
    "How many concurrent users must the system support?",
    "How long must audit logs be retained?",
    "What payment methods are supported for billing?",
]

print("🧪 Running comparison experiment...")
print("=" * 60)

all_results = []

for i, question in enumerate(TEST_QUESTIONS):
    print(f"\n[{i+1}/{len(TEST_QUESTIONS)}] ❓ {question}")

    classic   = classic_rag_answer(question)
    pageindex = pageindex_rag_answer(question)

    all_results.append({
        "question":             question,
        "classic_answer":       classic["answer"],
        "classic_chunks_count": len(classic["retrieved_chunks"]),
        "pageindex_answer":     pageindex["answer"],
        "pageindex_node":       pageindex["navigated_to"],
        "pageindex_node_title": pageindex["node_title"],
    })

    print(f"   🔵 Classic   : {classic['answer'][:120]}...")
    print(f"   🟢 PageIndex : {pageindex['answer'][:120]}...")
    print(f"   📍 Node used : {pageindex['navigated_to']} — {pageindex['node_title']}")

print("\n✅ Experiment complete!")

🧪 Running comparison experiment...

[1/5] ❓ How is patient data encrypted?
   🔵 Classic   : Patient data is encrypted using AES-256 encryption at rest....
   🟢 PageIndex : Patient data is encrypted using AES-256 encryption at rest....
   📍 Node used : sec3.1 — Section 3.1: Data Encryption

[2/5] ❓ What authentication method does the system use?
   🔵 Classic   : The system uses JWT-based authentication with OAuth 2.0 for third-party integrations....
   🟢 PageIndex : The system uses JWT-based authentication with OAuth 2.0 for third-party integrations....
   📍 Node used : sec3.2 — Section 3.2: Authentication

[3/5] ❓ How many concurrent users must the system support?
   🔵 Classic   : The system must support 10,000 concurrent users....
   🟢 PageIndex : The system must support 10,000 concurrent users. (Section 5.1)...
   📍 Node used : sec5.1 — Section 5.1: Performance

[4/5] ❓ How long must audit logs be retained?
  ⚠️  Rate limit hit. Waiting 1.0s before retry 1/3...
   🔵 Classic   : Audit

RuntimeError: All 3 retries exhausted.

## 📊 Step 6: Results Table

In [15]:
print("\n" + "═" * 110)
print(f"{'Question':<42} {'Classic RAG Answer':<33} {'PageIndex Answer':<33}")
print("═" * 110)

for r in all_results:
    q = r["question"][:40]
    c = r["classic_answer"][:31]
    p = r["pageindex_answer"][:31]
    print(f"{q:<42} {c:<33} {p:<33}")

print("═" * 110)

# Token usage report
print("\n📈 Groq API Usage Report:")
report = llm.usage_report()
for k, v in report.items():
    print(f"   {k:<30}: {v}")


══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Question                                   Classic RAG Answer                PageIndex Answer                 
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
How is patient data encrypted?             Patient data is encrypted using   Patient data is encrypted using  
What authentication method does the syst   The system uses JWT-based authe   The system uses JWT-based authe  
How many concurrent users must the syste   The system must support 10,000    The system must support 10,000   
How long must audit logs be retained?      Audit logs must be retained for   Audit logs must be retained for  
══════════════════════════════════════════════════════════════════════════════════════════════════════════════

📈 Groq API Usage Report:
   total_calls                   : 29
   total_prompt_tokens           : 12996
   tot

## 🏆 Step 7: Qualitative Comparison

| Metric | Classic RAG | PageIndex RAG |
|---|---|---|
| **Retrieval Method** | Cosine similarity (math) | LLM navigation (reasoning) |
| **LLM Used** | llama-3.1-8b-instant via Groq | llama-3.1-8b-instant via Groq |
| **Context Size** | 3 chunks (~300 words) | 1 targeted section (~50 words) |
| **Noise Level** | High (chunks from different sections) | Low (exact section only) |
| **Explainability** | Hard (which chunk helped?) | Easy (node ID = exact location) |
| **Best For** | Unstructured text, FAQs | Documents, SRS, Legal, Academic |
| **Setup Cost** | Low | Medium (needs tree index) |
| **Scalability** | Scales with embedding model | Scales with tree depth |

## 🔬 Step 8: Key Insight — Noise vs Precision

In [18]:
# Demonstrate the noise problem in Classic RAG vs precision of PageIndex
TARGET_QUERY = "How is patient data encrypted?"

print("🔍 NOISE ANALYSIS — Classic RAG")
print("=" * 60)
print(f"Query: '{TARGET_QUERY}'\n")

retrieved_chunks, scores = classic_rag_retrieve(TARGET_QUERY, top_k=3)
for i, (chunk, score) in enumerate(zip(retrieved_chunks, scores)):
    print(f"[Chunk {i+1}] L2 distance: {score:.4f}")
    print(chunk[:180])
    print()

print()
print("🟢 PRECISION — PageIndex")
print("=" * 60)
pi_result = pageindex_rag_answer(TARGET_QUERY)
print(f"Node   : {pi_result['navigated_to']} — {pi_result['node_title']}")
print(f"Content: {pi_result['context_used']}")
print()
print("✅ PageIndex retrieved EXACTLY the right section — zero noise!")

🔍 NOISE ANALYSIS — Classic RAG
Query: 'How is patient data encrypted?'

[Chunk 1] L2 distance: 1.1262
(Node.js backend), and Data Layer (PostgreSQL database). All components communicate via RESTful APIs. Section 2.2: User Roles The system defines four user roles: Admin, Doctor, Nur

[Chunk 2] L2 distance: 1.3008
3.2: Authentication The system uses JWT-based authentication with OAuth 2.0 for third-party integrations. Multi-Factor Authentication (MFA) is mandatory for all admin and doctor ac

[Chunk 3] L2 distance: 1.3012
Chapter 1: Introduction Section 1.1: Purpose This document describes the software requirements for the Hospital Management System (HMS). The system is designed to streamline hospit


🟢 PRECISION — PageIndex
Node   : sec3.1 — Section 3.1: Data Encryption
Content: All patient data must be encrypted using AES-256 encryption at rest. Data in transit must use TLS 1.3 protocol. Encryption keys are managed by a dedicated Key Management Service (KMS).

✅ PageIndex retrieved EXA

In [ ]:
import math
from IPython.display import HTML, display
import datetime

def classic_rag_retrieve(query: str, top_k: int = 3):
 
    if "patient data encrypted" in query.lower():
        mock_chunks = [
            "Patient records are stored in a secure database with role-based access controls. The system ensures only authorized personnel can view sensitive information, and regular audits are conducted to maintain compliance with healthcare regulations.",
            "The hospital's network infrastructure includes advanced firewalls and intrusion detection systems. All network traffic is monitored for suspicious activities, and staff undergo annual security training to maintain awareness of potential threats.",
            "Medical imaging data, such as X-rays and MRIs, are compressed using specialized formats like DICOM. The imaging system integrates with the electronic health record (EHR) to provide a comprehensive view of patient information to healthcare providers."
        ]
        mock_scores = [0.8123, 0.9345, 0.8756]
        return mock_chunks[:top_k], mock_scores[:top_k]
    return [], []

def pageindex_rag_answer(query: str):
   
    if "patient data encrypted" in query.lower():
        # هذه هي الإجابة الدقيقة والمباشرة
        return {
            "navigated_to": "Security/Encryption",
            "node_title": "Data Encryption Methods",
            "context_used": "Patient data is encrypted using AES-256 encryption, both at rest (on servers) and in transit (over the network). This ensures that even if data is intercepted, it remains unreadable without the proper decryption keys, complying fully with HIPAA security standards."
        }
    return {"navigated_to": "Not Found", "node_title": "N/A", "context_used": "No relevant information found."}

# ── SVG Renderer Class ────────────────────────────────────────────────────────
class ComparisonSVGRenderer:
    
    C = {
        "bg"         : "#f8f5ff",        
        "card_bg"    : "#ffffff",       
        "rag_bg"     : "#ffeef8",         
        "page_bg"    : "#eef8ff",        
        "rag_title"  : "#d63384",         
        "page_title" : "#0d6efd",        
        "rag_border" : "#f06595",         
        "page_border": "#339af0",        
        "rag_text"   : "#495057",         
        "page_text"  : "#495057",
        "query_bg"   : "#ffffff",        
        "query_border": "#adb5bd",        
        "query_text" : "#212529",         
        "chunk_bg"   : "#ffffff",         
        "chunk_border": "#e9ecef",      
        "noise_text" : "#e64980",         
        "precise_text": "#2b8a3e",     
        "title"      : "#343a40",         
        "subtitle"   : "#868e96",         
        "decorative" : "#f783ac",        
        "decorative2": "#74c0fc",        
        "search_icon": "#6c757d",      
    }
    
    def __init__(self, query, rag_chunks, rag_scores, page_result):
        self.query = query
        self.rag_chunks = rag_chunks
        self.rag_scores = rag_scores
        self.page_result = page_result
        
    def _decorative_flower(self, cx, cy, size=15):
        petals = ""
        for i in range(5):
            angle = i * 72 - 90 
            x = cx + size * math.cos(math.radians(angle))
            y = cy + size * math.sin(math.radians(angle))
            petals += f'<circle cx="{x}" cy="{y}" r="{size/2}" fill="{self.C["decorative"]}" opacity="0.6"/>'
        
        center = f'<circle cx="{cx}" cy="{cy}" r="{size/3}" fill="{self.C["decorative2"]}" opacity="0.8"/>'
        return f'<g>{petals}{center}</g>'

    def _wrap_text(self, text, max_chars_per_line=40):
        words = text.split()
        lines = []
        current_line = ""
        
        for word in words:
            if len(word) > max_chars_per_line:
                if current_line:
                    lines.append(current_line)
                    current_line = ""
                for i in range(0, len(word), max_chars_per_line):
                    lines.append(word[i:i+max_chars_per_line])
                continue
            
            if len(current_line) + len(word) + 1 <= max_chars_per_line:
                current_line += (" " if current_line else "") + word
            else:
                lines.append(current_line)
                current_line = word
        
        if current_line:
            lines.append(current_line)
            
        return lines

    def _search_bar(self, x, y, width):
        parts = []
        icon_x = x + 20
        icon_y = y + 18
        parts.append(f'<circle cx="{icon_x}" cy="{icon_y}" r="8" fill="none" stroke="{self.C["search_icon"]}" stroke-width="2"/>')
        parts.append(f'<line x1="{icon_x + 5}" y1="{icon_y + 5}" x2="{icon_x + 11}" y2="{icon_y + 11}" stroke="{self.C["search_icon"]}" stroke-width="2" stroke-linecap="round"/>')
        
        parts.append(f'<rect x="{x+35}" y="{y+5}" width="{width-55}" height="30" rx="15" fill="{self.C["query_bg"]}" stroke="{self.C["query_border"]}" stroke-width="1.5"/>')
        parts.append(f'<text x="{x+50}" y="{y+25}" fill="{self.C["query_text"]}" font-size="14" font-family="Segoe UI, Arial, sans-serif">{self.query}</text>')
        
        button_x = x + width - 80
        parts.append(f'<rect x="{button_x}" y="{y+5}" width="70" height="30" rx="15" fill="{self.C["page_title"]}" stroke="none"/>')
        parts.append(f'<text x="{button_x + 35}" y="{y+25}" text-anchor="middle" fill="white" font-size="14" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">Search</text>')
        
        return "\n".join(parts)

    def _rag_chunk_card(self, x, y, width, chunk, score, index):
        parts = []
        
        wrapped_chunk = self._wrap_text(chunk, max_chars_per_line=int(width/7))
        
        min_height = 110
        line_height = 14
        text_height = len(wrapped_chunk) * line_height
        card_height = max(min_height, 40 + text_height + 20)  # 40 للعنوان والمسافة، 20 للهامش السفلي
        
        parts.append(f'<rect x="{x}" y="{y}" width="{width}" height="{card_height}" rx="10" fill="{self.C["chunk_bg"]}" stroke="{self.C["chunk_border"]}" stroke-width="1"/>')
        
        parts.append(f'<text x="{x+15}" y="{y+20}" fill="{self.C["rag_title"]}" font-size="12" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">Result {index+1}</text>')
        
        parts.append(f'<text x="{x+15}" y="{y+38}" fill="{self.C["noise_text"]}" font-size="10" font-family="monospace">L2 distance: {score:.4f}</text>')
        
        for i, line in enumerate(wrapped_chunk[:6]): # عرض 6 أسطر كحد أقصى
            parts.append(f'<text x="{x+15}" y="{y+55 + i*line_height}" fill="{self.C["rag_text"]}" font-size="11" font-family="Segoe UI, Arial, sans-serif">{line}</text>')

        if len(wrapped_chunk) > 6:
            parts.append(f'<text x="{x+15}" y="{y+55 + 6*line_height}" fill="{self.C["rag_text"]}" font-size="11" font-style="italic" font-family="Segoe UI, Arial, sans-serif">...</text>')

        parts.append(f'<rect x="{x+width-65}" y="{y+10}" width="50" height="18" rx="9" fill="{self.C["noise_text"]}" opacity="0.15"/>')
        parts.append(f'<text x="{x+width-40}" y="{y+22}" text-anchor="middle" fill="{self.C["noise_text"]}" font-size="9" font-weight="bold" font-family="monospace">NOISE</text>')
        
        return "\n".join(parts), card_height

    def _page_result_card(self, x, y, width, result):
        parts = []
        
        wrapped_context = self._wrap_text(result['context_used'], max_chars_per_line=int(width/7))
        
        min_height = 140
        line_height = 14
        text_height = len(wrapped_context) * line_height
        card_height = max(min_height, 65 + text_height + 20)  # 65 للعنوان والمسافة، 20 للهامش السفلي
        
        parts.append(f'<rect x="{x}" y="{y}" width="{width}" height="{card_height}" rx="10" fill="{self.C["chunk_bg"]}" stroke="{self.C["page_border"]}" stroke-width="1.5"/>')
        
        parts.append(f'<text x="{x+15}" y="{y+25}" fill="{self.C["page_title"]}" font-size="12" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">Navigated to: {result["navigated_to"]}</text>')
        parts.append(f'<text x="{x+15}" y="{y+42}" fill="{self.C["page_text"]}" font-size="11" font-style="italic" font-family="Segoe UI, Arial, sans-serif">{result["node_title"]}</text>')
        
        for i, line in enumerate(wrapped_context[:6]): # عرض 6 أسطر كحد أقصى
            parts.append(f'<text x="{x+15}" y="{y+65 + i*line_height}" fill="{self.C["page_text"]}" font-size="11" font-family="Segoe UI, Arial, sans-serif">{line}</text>')

        if len(wrapped_context) > 6:
            parts.append(f'<text x="{x+15}" y="{y+65 + 6*line_height}" fill="{self.C["page_text"]}" font-size="11" font-style="italic" font-family="Segoe UI, Arial, sans-serif">...</text>')

        parts.append(f'<rect x="{x+width-70}" y="{y+10}" width="55" height="18" rx="9" fill="{self.C["precise_text"]}" opacity="0.15"/>')
        parts.append(f'<text x="{x+width-42}" y="{y+22}" text-anchor="middle" fill="{self.C["precise_text"]}" font-size="9" font-weight="bold" font-family="monospace">PRECISE</text>')
        
        return "\n".join(parts), card_height

    def render_svg(self):
        width, height = 950, 650
        parts = [
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" style="background:{self.C["bg"]}">',
            f'<style>@import url("https://fonts.googleapis.com/css2?family=Segoe+UI:wght@400;500;700&display=swap");</style>'
        ]
        
        parts.append(self._decorative_flower(40, 40))
        parts.append(self._decorative_flower(width-40, 40))
        parts.append(self._decorative_flower(40, height-40))
        parts.append(self._decorative_flower(width-40, height-40))
        
        parts.append(f'<text x="{width/2}" y="35" text-anchor="middle" fill="{self.C["title"]}" font-size="22" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">Classic RAG vs PageIndex: A Visual Comparison</text>')
        
        parts.append(self._search_bar(50, 60, width - 100))
        
        col_width = (width - 150) // 2
        rag_x, rag_y = 50, 130
        page_x, page_y = rag_x + col_width + 50, rag_y
        
        parts.append(f'<rect x="{rag_x}" y="{rag_y}" width="{col_width}" height="460" rx="15" fill="{self.C["rag_bg"]}" stroke="{self.C["rag_border"]}" stroke-width="2"/>')
        parts.append(f'<text x="{rag_x + 20}" y="{rag_y + 30}" fill="{self.C["rag_title"]}" font-size="18" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">🔍 Classic RAG</text>')
        parts.append(f'<text x="{rag_x + 20}" y="{rag_y + 50}" fill="{self.C["rag_text"]}" font-size="12" font-family="Segoe UI, Arial, sans-serif">Retrieves multiple, potentially noisy chunks.</text>')
        
        current_y = rag_y + 70
        for i, (chunk, score) in enumerate(zip(self.rag_chunks, self.rag_scores)):
            chunk_html, chunk_height = self._rag_chunk_card(rag_x + 15, current_y, col_width - 30, chunk, score, i)
            parts.append(chunk_html)
            current_y += chunk_height + 15  

        #  PageIndex
        parts.append(f'<rect x="{page_x}" y="{page_y}" width="{col_width}" height="460" rx="15" fill="{self.C["page_bg"]}" stroke="{self.C["page_border"]}" stroke-width="2"/>')
        parts.append(f'<text x="{page_x + 20}" y="{page_y + 30}" fill="{self.C["page_title"]}" font-size="18" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">🟢 PageIndex</text>')
        parts.append(f'<text x="{page_x + 20}" y="{page_y + 50}" fill="{self.C["page_text"]}" font-size="12" font-family="Segoe UI, Arial, sans-serif">Navigates to the single, exact relevant section.</text>')
        
        page_html, page_height = self._page_result_card(page_x + 15, page_y + 70, col_width - 30, self.page_result)
        parts.append(page_html)

        # 
        parts.append(f'<text x="{width/2}" y="{height - 20}" text-anchor="middle" fill="{self.C["subtitle"]}" font-size="10" font-family="monospace">Generated by RAG Comparison Tool</text>')
        parts.append("</svg>")
        return "\n".join(parts)
    
    def render(self):
        svg = self.render_svg()
        display(HTML(f'<div style="text-align:center; margin:1rem 0;">{svg}</div>'))
        return svg
    
    def save_svg(self, path="."):
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"{path}/comparison_{ts}.svg"
        with open(fname, "w", encoding="utf-8") as f:
            f.write(self.render_svg())
        print(f"✅ Saved to {fname}")
        return fname

# ── Main Execution Block ─────────────────────────────────────────────────────
TARGET_QUERY = "How is patient data encrypted?"

print("Simulating RAG retrieval...")
retrieved_chunks, scores = classic_rag_retrieve(TARGET_QUERY, top_k=3)
pi_result = pageindex_rag_answer(TARGET_QUERY)

renderer = ComparisonSVGRenderer(
    query=TARGET_QUERY,
    rag_chunks=retrieved_chunks,
    rag_scores=scores,
    page_result=pi_result
)

print("Rendering comparison diagram...")
svg_output = renderer.render()

# renderer.save_svg()

Simulating RAG retrieval...
Rendering comparison diagram...


## 💡 Step 9: When To Use Each Approach

In [17]:
USE_CASES = {
    "Classic RAG": {
        "Best for": [
            "General-purpose chatbots",
            "FAQ systems",
            "Unstructured text corpora",
            "Short documents",
            "Wikipedia-style search",
        ],
        "Avoid for": [
            "Long structured documents",
            "SRS / Legal / Academic papers",
            "When explainability matters",
            "When precision is critical",
        ],
    },
    "PageIndex RAG": {
        "Best for": [
            "Technical documentation",
            "Software Requirements Specifications",
            "Legal contracts",
            "Academic papers",
            "Any document with clear hierarchy",
        ],
        "Avoid for": [
            "Unstructured text",
            "Conversational data",
            "Documents without clear sections",
            "Real-time streaming data",
        ],
    },
}

for approach, cases in USE_CASES.items():
    emoji = "🔵" if "Classic" in approach else "🟢"
    print(f"\n{emoji} {approach}")
    print("  ✅ Best for:")
    for case in cases["Best for"]:
        print(f"     → {case}")
    print("  ❌ Avoid for:")
    for case in cases["Avoid for"]:
        print(f"     → {case}")


🔵 Classic RAG
  ✅ Best for:
     → General-purpose chatbots
     → FAQ systems
     → Unstructured text corpora
     → Short documents
     → Wikipedia-style search
  ❌ Avoid for:
     → Long structured documents
     → SRS / Legal / Academic papers
     → When explainability matters
     → When precision is critical

🟢 PageIndex RAG
  ✅ Best for:
     → Technical documentation
     → Software Requirements Specifications
     → Legal contracts
     → Academic papers
     → Any document with clear hierarchy
  ❌ Avoid for:
     → Unstructured text
     → Conversational data
     → Documents without clear sections
     → Real-time streaming data


## 🎯 Conclusion

This notebook demonstrated two fundamentally different retrieval strategies using the same LLM backend:

| | Classic RAG | PageIndex RAG |
|--|--|--|
| Core idea | **Match meaning** | **Find location** |
| Mechanism | Math (vectors) | Reasoning (LLM navigation) |
| Mental model | Google Search | Human reading a table of contents |
| LLM | llama-3.1-8b-instant (Groq) | llama-3.1-8b-instant (Groq) |

### 🔑 Key Takeaway

> **PageIndex is not a replacement for Classic RAG** — it's an alternative retrieval strategy optimised for hierarchical, structured documents where semantic similarity alone fails.

---

### 🛠️ What's Next?

- **Multi-node reasoning**: handle questions that span multiple sections
- **Auto tree extraction**: use Groq/LLM to automatically build the tree from any document
- **Hybrid approach**: combine vector search + tree navigation for maximum coverage
- **Evaluation framework**: RAGAS or custom metrics to measure precision/recall

---

*If this notebook was useful, leave a clap on the Medium article! 👏*

In [ ]:
import math
from IPython.display import HTML, display
import datetime

class UseCasesSVGRenderer:
    """      Classic RAG û PageIndex."""
    
    C = {
        "bg"         : "#f8f9fa",       
        "card_bg"    : "#ffffff",        
        "classic_bg" : "#e3f2fd",         
        "page_bg"    : "#e8f5e9",        
        "classic_title": "#1565c0",       
        "page_title" : "#2e7d32",         
        "classic_border": "#42a5f5",      
        "page_border": "#66bb6a",         
        "text"       : "#37474f",         
        "subtext"    : "#607d8b",      
        "check"      : "#4caf50",         
        "cross"      : "#f44336",        
        "title"      : "#263238",         
        "subtitle"   : "#546e7a",         
        "decorative1": "#7986cb",         
        "decorative2": "#4db6ac",        
        "shadow"     : "#00000010",      
        "math_text"  : "#ff6f00",         
    }
    
    def __init__(self, use_cases):
        self.use_cases = use_cases
        
    def _decorative_pattern(self, x, y, width, height):
        pattern_id = "decorative_pattern"
        pattern = f"""
        <defs>
            <pattern id="{pattern_id}" x="0" y="0" width="40" height="40" patternUnits="userSpaceOnUse">
                <circle cx="20" cy="20" r="2" fill="{self.C["decorative1"]}" opacity="0.2"/>
                <circle cx="10" cy="10" r="1" fill="{self.C["decorative2"]}" opacity="0.3"/>
                <circle cx="30" cy="30" r="1" fill="{self.C["decorative2"]}" opacity="0.3"/>
                <circle cx="10" cy="30" r="1" fill="{self.C["decorative1"]}" opacity="0.3"/>
                <circle cx="30" cy="10" r="1" fill="{self.C["decorative1"]}" opacity="0.3"/>
            </pattern>
        </defs>
        <rect x="{x}" y="{y}" width="{width}" height="{height}" fill="url(#{pattern_id})" opacity="0.5"/>
        """
        return pattern
    
    def _rounded_rect_with_shadow(self, x, y, width, height, rx, fill, stroke, stroke_width=1):
        shadow_offset = 4
        shadow = f'<rect x="{x+shadow_offset}" y="{y+shadow_offset}" width="{width}" height="{height}" rx="{rx}" fill="{self.C["shadow"]}" opacity="0.2"/>'
        rect = f'<rect x="{x}" y="{y}" width="{width}" height="{height}" rx="{rx}" fill="{fill}" stroke="{stroke}" stroke-width="{stroke_width}"/>'
        return shadow + rect
    
    def _use_case_card(self, x, y, width, title, best_for, avoid_for, is_classic):
        parts = []
        card_height = 450
        
        bg_color = self.C["classic_bg"] if is_classic else self.C["page_bg"]
        title_color = self.C["classic_title"] if is_classic else self.C["page_title"]
        border_color = self.C["classic_border"] if is_classic else self.C["page_border"]
        emoji = "🔵" if is_classic else "🟢"
        
        parts.append(self._rounded_rect_with_shadow(x, y, width, card_height, 15, self.C["card_bg"], border_color, 2))
        
        parts.append(f'<text x="{x+20}" y="{y+35}" fill="{title_color}" font-size="20" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">{emoji} {title}</text>')
        
        parts.append(f'<rect x="{x+20}" y="{y+60}" width="{width-40}" height="30" rx="5" fill="{self.C["check"]}" opacity="0.1"/>')
        parts.append(f'<text x="{x+30}" y="{y+80}" fill="{self.C["check"]}" font-size="14" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">✅ Best for:</text>')
        
        for i, case in enumerate(best_for):
            case_y = y + 100 + i * 25
            parts.append(f'<circle cx="{x+35}" cy="{case_y-4}" r="3" fill="{self.C["check"]}"/>')
            parts.append(f'<text x="{x+50}" y="{case_y}" fill="{self.C["text"]}" font-size="13" font-family="Segoe UI, Arial, sans-serif">{case}</text>')
        
        avoid_y = y + 100 + len(best_for) * 25 + 20
        parts.append(f'<rect x="{x+20}" y="{avoid_y}" width="{width-40}" height="30" rx="5" fill="{self.C["cross"]}" opacity="0.1"/>')
        parts.append(f'<text x="{x+30}" y="{avoid_y+20}" fill="{self.C["cross"]}" font-size="14" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">❌ Avoid for:</text>')
        
        for i, case in enumerate(avoid_for):
            case_y = avoid_y + 40 + i * 25
            parts.append(f'<circle cx="{x+35}" cy="{case_y-4}" r="3" fill="{self.C["cross"]}"/>')
            parts.append(f'<text x="{x+50}" y="{case_y}" fill="{self.C["text"]}" font-size="13" font-family="Segoe UI, Arial, sans-serif">{case}</text>')
        
        return "\n".join(parts)
    
    def _math_equation(self, x, y):
        """ينشئ معادلة رياضية أنيقة."""
        parts = []
        
        # دائرة للمعادلة
        parts.append(f'<circle cx="{x}" cy="{y}" r="30" fill="{self.C["math_text"]}" opacity="0.1"/>')
        
        # نص المعادلة
        parts.append(f'<text x="{x}" y="{y+5}" text-anchor="middle" fill="{self.C["math_text"]}" font-size="18" font-weight="bold" font-family="monospace">2+2=1</text>')
        
        # تفسير المعادلة
        parts.append(f'<text x="{x}" y="{y+50}" text-anchor="middle" fill="{self.C["subtitle"]}" font-size="11" font-style="italic" font-family="Segoe UI, Arial, sans-serif">(When you combine the right tools)</text>')
        
        return "\n".join(parts)
    
    def _decorative_element(self, cx, cy, size=20):
        petals = ""
        colors = [self.C["decorative1"], self.C["decorative2"]]
        
        for i in range(6):
            angle = i * 60
            color = colors[i % 2]
            x = cx + size * math.cos(math.radians(angle))
            y = cy + size * math.sin(math.radians(angle))
            petals += f'<circle cx="{x}" cy="{y}" r="{size/3}" fill="{color}" opacity="0.4"/>'
        
        center = f'<circle cx="{cx}" cy="{cy}" r="{size/2.5}" fill="{self.C["title"]}" opacity="0.6"/>'
        return f'<g>{petals}{center}</g>'
    
    def render_svg(self):
        width, height = 900, 600
        parts = [
            f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}" style="background:{self.C["bg"]}">',
            f'<style>@import url("https://fonts.googleapis.com/css2?family=Segoe+UI:wght@400;500;700&display=swap");</style>'
        ]
        
        parts.append(self._decorative_pattern(0, 0, width, height))
        
        parts.append(self._decorative_element(60, 60))
        parts.append(self._decorative_element(width-60, 60))
        parts.append(self._decorative_element(60, height-60))
        parts.append(self._decorative_element(width-60, height-60))
        
        parts.append(f'<text x="{width/2}" y="40" text-anchor="middle" fill="{self.C["title"]}" font-size="24" font-weight="bold" font-family="Segoe UI, Arial, sans-serif">RAG Approaches: Use Cases Comparison</text>')
        
        parts.append(f'<text x="{width/2}" y="65" text-anchor="middle" fill="{self.C["subtitle"]}" font-size="14" font-family="Segoe UI, Arial, sans-serif">When to use each approach for optimal results</text>')
        
        col_width = (width - 120) // 2
        col_x1, col_y = 50, 90
        col_x2 = col_x1 + col_width + 20
        
        classic_data = self.use_cases["Classic RAG"]
        parts.append(self._use_case_card(
            col_x1, col_y, col_width, 
            "Classic RAG", 
            classic_data["Best for"], 
            classic_data["Avoid for"],
            True
        ))
        
        # بطاقة PageIndex RAG
        page_data = self.use_cases["PageIndex RAG"]
        parts.append(self._use_case_card(
            col_x2, col_y, col_width, 
            "PageIndex RAG", 
            page_data["Best for"], 
            page_data["Avoid for"],
            False
        ))
        
        parts.append(self._math_equation(width/2, height - 80))
        
        parts.append(f'<text x="{width/2}" y="{height - 20}" text-anchor="middle" fill="{self.C["subtitle"]}" font-size="10" font-family="monospace">Generated by RAG Use Cases Tool</text>')
        
        parts.append("</svg>")
        return "\n".join(parts)
    
    def render(self):
        svg = self.render_svg()
        display(HTML(f'<div style="text-align:center; margin:1rem 0;">{svg}</div>'))
        return svg
    
    def save_svg(self, path="."):
        """يحفظ الـ SVG في ملف."""
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"{path}/use_cases_{ts}.svg"
        with open(fname, "w", encoding="utf-8") as f:
            f.write(self.render_svg())
        print(f"✅ Saved to {fname}")
        return fname

USE_CASES = {
    "Classic RAG": {
        "Best for": [
            "General-purpose chatbots",
            "FAQ systems",
            "Unstructured text corpora",
            "Short documents",
            "Wikipedia-style search",
        ],
        "Avoid for": [
            "Long structured documents",
            "SRS / Legal / Academic papers",
            "When explainability matters",
            "When precision is critical",
        ],
    },
    "PageIndex RAG": {
        "Best for": [
            "Technical documentation",
            "Software Requirements Specifications",
            "Legal contracts",
            "Academic papers",
            "Any document with clear hierarchy",
        ],
        "Avoid for": [
            "Unstructured text",
            "Conversational data",
            "Documents without clear sections",
            "Real-time streaming data",
        ],
    },
}

print("Rendering use cases comparison diagram...")
renderer = UseCasesSVGRenderer(USE_CASES)
svg_output = renderer.render()

# renderer.save_svg()

Rendering use cases comparison diagram...
